# Predição - Classificação: Prova de Fogo - Jaqueta

Autor:  Viviane da Rosa Sommer

Atualizado: 21/02/2024

## Objetivo:
Notebook para fazer a predição do modelo de Coral-Sol (V3 - Classificação) nos vídeos de Jaqueta.

Essa versão gerar um vídeo com os frames originais junto de suas probabilidades.
Ao final, também é gerado um gráfico e CSV , relacionado aos Scores (Probabilidade) vs Tempo.

## Importações Necessárias

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import math
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor

## Declaração de Constantes e Modelos

In [ ]:
input_folder = "Videos"
output_folder = "Videos-Resultado-Class"
model_v2 = YOLO('../Coral-Classificacao/Pesos/best_weights_v3.pt')

## Funções Necessárias

In [ ]:
def plot_from_csv(csv_path: str, output_plot_path: str, video_name: str) -> None:
    """
    Reads a CSV file and generates a confidence-over-time plot for positive and negative classes.

    Parameters:
    csv_path (str): Path to the input CSV file.
    output_plot_path (str): Path to save the output plot.
    video_name (str): Name of the video being processed.

    Returns:
    None
    """
    df = pd.read_csv(csv_path)

    if not {"Time (s)", "Positive Confidence", "Negative Confidence"}.issubset(df.columns):
        print(f"Invalid CSV or missing columns: {csv_path}")
        return

    df['Time (s)'] = df['Time (s)'].apply(math.floor)

    grouped = df.groupby('Time (s)')[['Positive Confidence', 'Negative Confidence']].max().reset_index()

    plt.figure(figsize=(10, 6))
    plt.plot(grouped['Time (s)'], grouped['Positive Confidence'], label='Positive Confidence', color='g', linewidth=2)
    plt.plot(grouped['Time (s)'], grouped['Negative Confidence'], label='Negative Confidence', color='r', linewidth=2)
    plt.xlabel('Time (s)')
    plt.ylabel('Confidence')
    plt.title(f'Confidence Over Time\nVideo: {video_name}')
    plt.ylim(0, 1)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()

    plt.savefig(output_plot_path)
    plt.close()
    print(f"Plot saved at: {output_plot_path}")


def annotate_frame_with_predictions(frame: np.ndarray, positive_score: float, negative_score: float) -> np.ndarray:
    """
    Adds annotations to a video frame, displaying positive and negative scores.

    Parameters:
    frame (np.ndarray): Video frame.
    positive_score (float): Positive class confidence score.
    negative_score (float): Negative class confidence score.

    Returns:
    np.ndarray: Annotated video frame.
    """
    annotated_frame = frame.copy()

    height, _, _ = frame.shape
    start_x = 20
    center_y = height // 2

    positive_text = f"Positive: {positive_score:.2f}"
    negative_text = f"Negative: {negative_score:.2f}"

    positive_text_position = (start_x, center_y - 20)
    negative_text_position = (start_x, center_y + 20)

    cv2.putText(annotated_frame, positive_text, positive_text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.8, color=(0, 255, 0), thickness=2)
    cv2.putText(annotated_frame, negative_text, negative_text_position, cv2.FONT_HERSHEY_SIMPLEX, 0.8, color=(0, 0, 255), thickness=2)

    return annotated_frame


def process_frame(frame_idx: int, frame: np.ndarray, fps: int, model) -> dict:
    """
    Processes a single video frame for inference and annotation.

    Parameters:
    frame_idx (int): Index of the video frame.
    frame (np.ndarray): Video frame.
    fps (int): Frames per second of the video.
    model: Classification model.

    Returns:
    dict: Inference results and annotated frame.
    """
    timestamp = frame_idx / fps

    result = model(frame, verbose=False)
    positive_score = result[0].probs.data.cpu().numpy()[1]
    negative_score = result[0].probs.data.cpu().numpy()[0]

    annotated_frame = annotate_frame_with_predictions(frame, positive_score, negative_score)

    return {
        'Frame': frame_idx,
        'Time (s)': timestamp,
        'Positive Confidence': positive_score,
        'Negative Confidence': negative_score,
        'Annotated Frame': annotated_frame
    }


def process_video_parallel_limited_memory(video_path: str, csv_output_path: str, video_output_path: str,
                                          plot_path: str, model, batch_size: int = 16, num_workers: int = 4) -> None:
    """
    Processes a video using parallelism with memory control (batch processing).

    Parameters:
    video_path (str): Path to the input video.
    csv_output_path (str): Path to save the CSV file.
    video_output_path (str): Path to save the annotated video.
    plot_path (str): Path to save the plot.
    model: Classification model.
    batch_size (int): Number of frames processed per batch.
    num_workers (int): Number of parallel threads.

    Returns:
    None
    """
    import gc

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(video_output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    results = []
    frames = []

    for frame_idx in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            break
        frames.append((frame_idx, frame))

        if len(frames) == batch_size or frame_idx == total_frames - 1:
            with ThreadPoolExecutor(max_workers=num_workers) as executor:
                futures = [executor.submit(process_frame, frame_idx, frame, fps, model) for frame_idx, frame in frames]

            for future in futures:
                result = future.result()
                results.append({
                    'Frame': result['Frame'],
                    'Time (s)': result['Time (s)'],
                    'Positive Confidence': result['Positive Confidence'],
                    'Negative Confidence': result['Negative Confidence']
                })
                out.write(result['Annotated Frame'])

            frames = []
            gc.collect()

    cap.release()
    out.release()

    pd.DataFrame(results).to_csv(csv_output_path, index=False)
    print(f"Results saved at: {csv_output_path}")
    print(f"Annotated video saved at: {video_output_path}")

    plot_from_csv(csv_output_path, plot_path, os.path.basename(video_path))


def process_all_videos_in_folder(input_folder: str, output_folder: str, model) -> None:
    """
    Processes all videos in a folder, generating annotated videos, CSV files, and plots.

    Parameters:
    input_folder (str): Folder containing input videos.
    output_folder (str): Folder to save the outputs.
    model: Classification model.

    Returns:
    None
    """
    os.makedirs(output_folder, exist_ok=True)
    video_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.mp4', '.avi', '.mov'))]
    print(f"Videos found in folder '{input_folder}': {len(video_files)}")

    for video_file in tqdm(video_files, desc="Processing videos"):
        video_path = os.path.join(input_folder, video_file)
        base_name = os.path.splitext(video_file)[0]

        csv_output_path = os.path.join(output_folder, f"{base_name}_results.csv")
        video_output_path = os.path.join(output_folder, f"{base_name}_output.mp4")
        plot_path = os.path.join(output_folder, f"{base_name}_plot.png")

        print(f"Processing video: {video_file}")

        process_video_parallel_limited_memory(
            video_path=video_path,
            csv_output_path=csv_output_path,
            video_output_path=video_output_path,
            plot_path=plot_path,
            model=model,
            batch_size=16,
            num_workers=4
        )

    print(f"Processing complete! Results saved in: {output_folder}")


## Processa vídeos

In [ ]:
process_all_videos_in_folder(
    input_folder=input_folder,
    output_folder=output_folder,
    model=model_v2
)

In [ ]:
!jupyter nbconvert --to html Class_Video.ipynb